# S-GMS

Monte-Carlo dropout voting over the candidate diagrams, then selection of the
representative records.

In [ ]:
import os, glob
import numpy as np

from nets.MobileNetV3_m import build_classifier
from nets.selectwave import select_group, select_all
from utils.dataloader import make_predict_dataset, list_images

RDTD_DIR   = 'data/rdtd_labeled/'
# newest finished training run (cell 2 raises if none exists)
_hits = sorted(glob.glob('runs/*/final.weights.h5'))
WEIGHTS    = _hits[-1] if _hits else 'runs/final.weights.h5'
CURVE_DIR  = 'data/midr/'
TARGET_DIR = 'data/target/'
OUT        = 'output/'

CHANNELS = 3
PASSES   = 30     # forward passes per pool
TOP_N    = 2      # records retained per class

for d in (RDTD_DIR, CURVE_DIR, TARGET_DIR, OUT):
    state = 'ok' if os.path.isdir(d) else \
            'missing (OUT is created on demand, the others are not)'
    print(f'[info] {d:<24}{state}')
print(f'[info] weights     {WEIGHTS}')


In [ ]:
paths, classes = list_images(RDTD_DIR)
label_map = {n: i for i, n in enumerate(classes)}   # folder -> class index (alphabetical)
ds = make_predict_dataset(paths, size=(224, 224))

model = build_classifier(CHANNELS, 1, 128, 0.2)
if not os.path.exists(WEIGHTS):
    raise FileNotFoundError(
        f'{WEIGHTS} not found. Run S-train first, or point WEIGHTS at a '
        'runs/<timestamp>/final.weights.h5 file - voting with random weights '
        'would silently produce meaningless selections.')
model.load_weights(WEIGHTS)


In [ ]:
def curve_of(path):
    """Response curve belonging to one record.

    Expects one ``<group>_<record>.txt`` file per record inside ``CURVE_DIR``
    (the per-floor nonlinear response of that record).
    """
    key = os.path.splitext(os.path.basename(str(path)))[0]
    hit = glob.glob(os.path.join(CURVE_DIR, key + '*.txt'))  # <record>*.txt
    return np.loadtxt(hit[0]) if hit else None


def target_of(group):
    """Target quantiles of one group: ``<group>_16.txt`` / ``<group>_84.txt``."""
    lo = glob.glob(os.path.join(TARGET_DIR, group + '_16.txt')) or          glob.glob(os.path.join(TARGET_DIR, group + '*16*.txt'))
    hi = glob.glob(os.path.join(TARGET_DIR, group + '_84.txt')) or          glob.glob(os.path.join(TARGET_DIR, group + '*84*.txt'))
    if not (lo and hi):
        print(f'[warn] target quantiles missing for group {group}; DRC skipped')
        return None, None
    return np.loadtxt(lo[0]), np.loadtxt(hi[0])


In [ ]:
group = None    # restrict to one group, or None for all records
subset = paths if group is None else [p for p in paths if group in p]

rec = select_group(model, ds, subset, curve_of, label_map=label_map,
                   passes=PASSES, top_n=TOP_N)

for side in ('pos', 'neg'):
    r = rec[side]
    print(f"{side}: pool {r['n_pool']}  selected {len(r['paths'])}  "
          f"votes {r['freq']}/{PASSES}")


In [ ]:
table = select_all(model, ds, paths, curve_of, target_of, label_map=label_map,
                   passes=PASSES, top_n=TOP_N, out_dir=OUT)
table.head()
